# 🔧 題目 1：零售 POS 銷售分析
# Mini Data Pipeline 工作坊

> **情境**：你是一家零售連鎖集團的資料顧問。老闆想知道哪些商品最暢銷、哪些客戶最有價值、各國市場表現如何。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI`
>
> **資料**：[Kaggle: Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`

---

### 📋 今日目標

| 必做（Section 1-8） | 回家作業（Section 9-10） |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | 🏠 Dashboard（回家作業） |
| ✅ LLM 分析 + Pipeline Documentation | |
| ✅ LLM 品類分類 | ⭐ 本地部署 |
| ✅ Pipeline Documentation | |

### 🗺️ 標記說明

| 標記 | 意思 |
|------|------|
| `🟢 簡單` | 開放式 — 提示裡有範例教語法，自己應用到這題 |
| `🟡 中等` | 半骨架 — 用別的情境示範，你「翻譯」到自己的欄位 |
| `🔴 較難` | 完整骨架 — 結構都給了，填入關鍵的欄位名和 SQL |
| `（不需要改）` | 直接跑 |


## Section 0：環境設定

直接跑，不需要改。


In [ ]:
# （不需要改）Colab 環境自動設定
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('topic_1'):
        !git clone https://github.com/lu791019/midterm-mvp-template.git /content/repo
    os.chdir('/content/repo/data/raw/topic_1')
    print("✅ Colab：已設定工作目錄 =", os.getcwd())
else:
    print("✅ 本地環境，工作目錄 =", os.getcwd())


In [ ]:
# （不需要改）
import pandas as pd
import sqlite3
import os
import json
print("✅ 套件載入完成")


In [ ]:
# （不需要改）
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback（不影響完成度）")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> pipeline 第一步：**資料進入系統**。


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **raw 表**（原始資料全部灌入）
- 知道資料有幾筆、幾欄、什麼型別

> ⏱ 時間不夠？只做 Step 1-1 和 1-4（讀 CSV + 存入 raw 表），跳過探索。


### Step 1-1 🟢 簡單：讀取 CSV

> 💡 `pd.read_csv()` 把 CSV 檔案讀成 DataFrame（一個表格）
> 💡 **範例**：如果要讀一份學生名單：
> ```python
> students = pd.read_csv("data/students.csv")
> print(f"共 {len(students)} 筆")
> print(list(students.columns))
> students.head()
> ```
> 🎯 現在對 `orders.csv` 做同樣的事
> ✅ 預期：2,000 筆、9 個欄位


In [ ]:
# TODO 🟢: 讀取 orders.csv，印出筆數、欄位、前 5 筆

# 相關程式碼：
# df_raw = pd.read_csv("檔案路徑")
# print(f"共 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
# print(f"欄位: {list(df_raw.columns)}")
# df_raw.head()


### Step 1-2 🟢 簡單：檢查資料品質

> 💡 拿到新資料，先做三件事：
> ```python
> # 範例：檢查學生名單
> print(students.dtypes)         # 每個欄位是什麼型別（數字？文字？日期？）
> print(students.isnull().sum()) # 哪些欄位有空值、各幾筆
> print(students.describe())     # 數值欄位的最小/最大/平均
> ```
> 🎯 對你的 df_raw 做同樣三件事
> ✅ 預期：quantity 和 unit_price 是數值，invoice_date 是字串


In [ ]:
# TODO 🟢: 用三個方法檢查 df_raw

# 相關程式碼：
# print(df_raw.dtypes)           # 看型別
# print(df_raw.isnull().sum())   # 看缺漏值
# print(df_raw.describe())       # 看數值統計


### Step 1-3 🟢 簡單：自由探索

> 💡 探索資料的常用招式：
> ```python
> df["欄位"].value_counts()     # 某欄位各值出現幾次（例如國家分佈）
> df["欄位"].nunique()           # 某欄位有幾種不同的值
> df.sample(5)                   # 隨機看 5 筆
> df.groupby("欄位")["數值欄"].mean()  # 分組看平均
> ```
> 🎯 用上面的招式探索 df_raw，了解資料長什麼樣


In [ ]:
# TODO 🟢: 自由探索 — 看國家分佈？看商品有幾種？看金額範圍？

# 相關程式碼：
# df_raw["country"].value_counts()     # 各國有幾筆
# df_raw["description"].nunique()       # 有幾種商品
# df_raw["total_amount"].describe()     # 金額統計
# df_raw.sample(5)                      # 隨機看 5 筆


### Step 1-4 🟡 中等：建立 SQLite + 寫入 raw 表

> 💡 **範例**：假設要把學生名單存進資料庫：
> ```python
> # 1. 建立資料庫連線（檔案不存在會自動建立）
> conn = sqlite3.connect("school.db")
>
> # 2. 把 DataFrame 寫入資料庫的 "students" 表
> students.to_sql("students", conn, if_exists="replace", index=False)
>
> # 3. 用 SQL 驗證有沒有成功
> result = pd.read_sql("SELECT COUNT(*) as total FROM students", conn)
> print(f"students 表: {result['total'][0]} 筆")
> ```
> 🎯 現在建立 `pipeline.db`，把 df_raw 寫入 `raw_orders` 表
> ✅ 預期：「raw_orders: 2000 筆」


In [ ]:
# TODO 🟡: 建立 SQLite，寫入 raw 表，驗證
DB_PATH = "pipeline.db"

# 相關程式碼：
# conn = sqlite3.connect(DB_PATH)
# df_raw.to_sql("raw_orders", conn, if_exists="replace", index=False)
# result = pd.read_sql("SELECT COUNT(*) as total FROM raw_orders", conn)
# print(f"✅ raw_orders: {result['total'][0]} 筆")


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> 從**資料庫**讀出 → 清洗 → 寫回資料庫。不是從 CSV 讀！


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **cleaned 表**（已清洗）
- 清洗邏輯包含：去缺值、轉型別、必要時新增欄位

> ⏱ 時間不夠？只做 Step 2-1、2-2 和 2-5（讀出 + 基本清洗 + 存入 cleaned 表），跳過進階處理。


### Step 2-1 🟢 簡單：從 raw 表讀出

> 💡 **範例**：從資料庫讀出學生名單：
> ```python
> students = pd.read_sql("SELECT * FROM students", conn)
> print(f"讀出 {len(students)} 筆")
> ```
> 🎯 從 `raw_orders` 表讀出資料，存成 `df`
> ✅ 預期：df 有 2000 筆


In [ ]:
# TODO 🟢: 從 raw_orders 讀出，記下清洗前筆數


### Step 2-2 🟢 簡單：處理缺漏值

> 💡 **範例**：刪除「姓名」或「成績」為空的學生：
> ```python
> students = students.dropna(subset=["name", "score"])
> ```
> 🎯 刪除 `description` 或 `customer_id` 為空的列
> ✅ 預期：筆數可能減少幾筆


In [ ]:
# TODO 🟢: 刪除缺漏值，印出前後筆數


### Step 2-3 🟡 中等：日期轉換 + 新增時間特徵

> 💡 **範例**：假設有出貨紀錄，想從日期提取年月：
> ```python
> # 先把文字轉成日期型別
> ship_df["ship_date"] = pd.to_datetime(ship_df["ship_date"])
>
> # 再從日期提取特徵
> ship_df["ship_year"] = ship_df["ship_date"].dt.year
> ship_df["ship_month"] = ship_df["ship_date"].dt.month
> ship_df["ship_weekday"] = ship_df["ship_date"].dt.day_name()
> ship_df["ship_hour"] = ship_df["ship_date"].dt.hour
> ```
> 🎯 對 `invoice_date` 做同樣的事，新增 `year`, `month`, `day_of_week`, `hour`
> ✅ 預期：df 多了 4 個新欄位


In [ ]:
# TODO 🟡: 日期轉換 + 提取 4 個時間特徵


### Step 2-4 🟢 簡單：計算總金額 + 過濾異常值

> 💡 **範例**：計算學生的加權總分，過濾不合理的值：
> ```python
> students["total"] = students["midterm"] * 0.4 + students["final"] * 0.6
> students = students[students["total"] >= 0]  # 過濾負分
> ```
> 🎯 確認 `total_amount = quantity × unit_price`，過濾 quantity ≤ 0 或 unit_price ≤ 0
> ✅ 預期：total_amount 全是正數


In [ ]:
# TODO 🟢: 計算總金額 + 過濾異常值


### 🏁 清洗檢查點

> 直接跑。全部 ✅ 才往下。


In [ ]:
# （不需要改）
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值！回去看 Step 2-2"
assert (df["quantity"] > 0).all(), "❌ quantity 有非正值！回去看 Step 2-4"
assert (df["unit_price"] > 0).all(), "❌ unit_price 有非正值！回去看 Step 2-4"
assert "year" in df.columns, "❌ 缺少 year！回去看 Step 2-3"
assert "month" in df.columns, "❌ 缺少 month！回去看 Step 2-3"
print("✅ 全部檢查通過！")
print(f"   清洗後: {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5 🟢 簡單：寫入 cleaned 表

> 💡 跟 Step 1-4 一樣用 `to_sql`：
> ```python
> df_clean.to_sql("cleaned_表名", conn, if_exists="replace", index=False)
> print(f"✅ cleaned: {len(df_clean)} 筆")
> ```
> 💡 表名改成 `cleaned_orders`
> ✅ 預期：cleaned_orders 筆數 ≤ raw_orders


In [ ]:
# TODO 🟢: 寫入 cleaned_orders 表，驗證兩表筆數


---
## Section 3：統計分析（pandas / SQL）

> 用 pandas 或 SQL 從資料庫查詢統計。


### 🎯 完成這段後你應該有：
- 至少一張統計表（`processed/*.csv`）
- 至少一個可以在 Demo 裡講的數字

> ⏱ 時間不夠？只做 Step 3-1（一個 GROUP BY 查詢），跳過視覺化和自由探索。


### Step 3-1 🟡 中等：商品銷售排行

> 💡 **範例**：假設要從學生表查各班平均成績排行：
> ```python
> class_stats = pd.read_sql("""
>     SELECT class_name,
>            COUNT(*) as student_count,
>            ROUND(AVG(score), 2) as avg_score
>     FROM students
>     GROUP BY class_name
>     ORDER BY avg_score DESC
>     LIMIT 10
> """, conn)
> ```
> 🎯 從 `cleaned_orders` 查商品銷售排行：description, 訂單數, 總數量, 總金額 Top 20
> 💡 用 `SUM(total_amount)` 算營收、`SUM(quantity)` 算總量
> ✅ 預期：看到最暢銷的 20 個商品


In [ ]:
# TODO 🟡: 商品銷售排行 SQL

# Hint: GROUP BY description，用 SUM(quantity) 和 SUM(total_amount)
# Skeleton:
# product_stats = pd.read_sql("""
#     SELECT description, COUNT(*) AS order_count,
#            SUM(quantity) AS total_qty,
#            ROUND(SUM(total_amount),2) AS revenue
#     FROM cleaned_orders
#     GROUP BY description ORDER BY revenue DESC LIMIT 20
# """, conn)

product_stats = pd.read_sql("""

""", conn)
product_stats


### Step 3-2 🟡 中等：各國銷售統計

> 💡 **範例**：查各班男女人數（不重複計算）：
> ```python
> pd.read_sql("""
>     SELECT class_name,
>            COUNT(DISTINCT student_id) as unique_students,
>            COUNT(*) as total_records
>     FROM students
>     GROUP BY class_name
> """, conn)
> ```
> 🎯 從 `cleaned_orders` 查各國：不重複客戶數、訂單數、營收
> 💡 用 `COUNT(DISTINCT customer_id)` 算客戶數
> ✅ 預期：United Kingdom 營收最高


In [ ]:
# TODO 🟡: 各國銷售統計 SQL
country_stats = pd.read_sql("""

""", conn)
country_stats


### Step 3-3 🟡 中等：視覺化

> 💡 **範例**：把班級成績畫成橫條圖：
> ```python
> import matplotlib.pyplot as plt
> class_stats.plot.barh(x="class_name", y="avg_score", figsize=(10, 5))
> plt.title("各班平均成績")
> plt.tight_layout()
> plt.show()
> ```
> 🎯 把 Step 3-1 或 3-2 的結果畫成圖
> ✅ 預期：至少一張圖表


In [ ]:
# TODO 🟡: 視覺化
import matplotlib.pyplot as plt


### Step 3-4 🟢 簡單：自由探索 SQL

> 💡 靈感和對應的 SQL 寫法：
> ```sql
> -- 每月銷售趨勢
> SELECT year, month, SUM(total_amount) FROM cleaned_orders GROUP BY year, month
>
> -- 尖峰時段
> SELECT hour, COUNT(*) FROM cleaned_orders GROUP BY hour ORDER BY COUNT(*) DESC
>
> -- VIP 客戶
> SELECT customer_id, SUM(total_amount) FROM cleaned_orders GROUP BY customer_id ORDER BY SUM(total_amount) DESC LIMIT 10
> ```
> 🎯 選一個你好奇的問題，寫 SQL 查


In [ ]:
# TODO 🟢: 你自己的 SQL


### Step 3-5 🟢 簡單：存統計結果

> 💡 **範例**：
> ```python
> os.makedirs("results", exist_ok=True)  # exist_ok=True 表示已存在不報錯
> class_stats.to_csv("results/class_stats.csv", index=False)
> ```
> 🎯 建立 `data/processed/` 資料夾，把統計結果存成 CSV


In [ ]:
# TODO 🟢: 存統計結果


### 💡 你還可以分析什麼？（進階探索）

- 📊 **月/季營收趨勢**：哪幾個月賣最好？有沒有季節性？
- 👥 **客戶價值分析**：哪些客戶買最多？消費金額 Top 10？
- 🕐 **時段分析**：幾點下單最多？（用 invoice_date 的小時）
- 🌍 **國家比較**：各國平均客單價差異？
- 🔄 **商品組合**：同一張訂單常一起買的商品有哪些？

> 以上都可以用一個 SQL `GROUP BY` 搞定。挑一個有趣的，Demo 時講。


---
## Section 4：LLM 加值分析

> helper 函式已寫好（API 呼叫太複雜），你要做的是：呼叫它、看結果、跑批次、寫入資料庫。


### 🎯 完成這段後你應該有：
- `pipeline.db` 裡有一張 **analyzed 表**（Bronze/Silver/Gold 三表齊全）
- LLM 或 fallback 分析結果寫入 analyzed 表

> ⏱ 沒有 API Key？直接用 fallback 規則版，一樣能完成。


In [ ]:
# （不需要改）LLM Helper
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}
商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["christmas","xmas","santa","winter"]): cat = "季節商品"
    elif any(w in t for w in ["candle","holder","frame","lamp"]): cat = "家飾"
    elif any(w in t for w in ["cup","mug","plate","bowl"]): cat = "餐具"
    elif any(w in t for w in ["pen","pencil","notebook","card"]): cat = "文具"
    elif any(w in t for w in ["gift","bag","box","ribbon"]): cat = "禮品"
    else: cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper 已定義")


# 批次分析 helper（不需要改）
def run_batch_analysis(df, text_column, conn, table_name, n=50, api_key=None):
    """一行搞定：批次 LLM 分析 + 寫入 analyzed 表。"""
    df_batch = df.head(n).copy()
    results = []
    for i, row in df_batch.iterrows():
        r = llm_analyze(str(row[text_column]), api_key)
        results.append(r)
        if len(results) % 10 == 0:
            print(f"  進度: {len(results)}/{n}")
    df_batch["category"] = [r.get("category", "") for r in results]
    df_batch["llm_insight"] = [r.get("insight", "") for r in results]
    df_batch.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"✅ {table_name}: {len(df_batch)} 筆已寫入")
    return df_batch



### Step 4-1 🟢 簡單：單筆測試

> 💡 **範例**：
> ```python
> # 取第一筆學生的作文
> essay = students["essay"].iloc[0]
> # 呼叫分析
> result = analyze(essay)
> print(result)
> ```
> 🎯 取 df 的第一筆 `description`，呼叫 `llm_analyze(文字, api_key)`
> 💡 api_key 傳 `OPENAI_API_KEY if OPENAI_API_KEY else None`
> ✅ 預期：回傳 dict 有 `category` 和 `insight`


In [ ]:
# TODO 🟢: 單筆測試


### Step 4-2 🟢 簡單：批次分析 + 寫入 analyzed 表

> 💡 上面的 `run_batch_analysis()` 幫你一行搞定：批次呼叫 LLM + 整理結果 + 寫入資料庫
>
> 🎯 呼叫 `run_batch_analysis(df, "description", conn, "analyzed_orders")` 
> ✅ 預期：`analyzed_orders` 表有 50 筆，多了 `category` 和 `llm_insight` 欄位


In [ ]:
# TODO 🟢: 一行搞定批次 LLM 分析
df_analyzed = run_batch_analysis(df, "description", conn, "analyzed_orders", n=50, api_key=OPENAI_API_KEY if OPENAI_API_KEY else None)
df_analyzed.head()


### ✅ 檢查點：三表驗證

> 跑完下面這格確認三張表都有資料。


In [ ]:
# TODO 🟡: 跨表查詢驗證
lineage = pd.read_sql("""

""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))


---
## Section 6：Pipeline Documentation


### 🎯 完成這段後你應該有：
- `output/pipeline_doc.md` 有 Pipeline Documentation（用 AI prompt 快速產出）

> ⏱ 時間不夠？先用 AI prompt 產草稿，課後再補完整版。
> 📋 課後把這份文件整理成你自己 repo 的 README.md。


### Step 6-1 🟢 簡單：寫報告

> 💡 **範例**：用 f-string 嵌入數字：
> ```python
> avg = pd.read_sql("SELECT AVG(score) as a FROM students", conn)["a"][0]
> report = f"平均成績是 {avg:.1f} 分"
> ```
> 🎯 用你在 Section 3-4 的分析結果，填入報告模板
> 💡 先用 SQL 查出需要的數字（總銷售額等），再嵌入 f-string
> ✅ 預期：output/pipeline_doc.md 有具體數字和建議


In [ ]:
# TODO 🟢: 寫報告
report = f"""# 零售 POS 銷售分析報告

## 資料概要
（填入：分析筆數、總銷售額、資料來源）

## 關鍵發現
（根據 Section 3 統計，寫 2-3 個有數字的發現）

## 品類分佈
（根據 Section 4 LLM 分析，列出分佈）

## 建議
（寫 2-3 條有數據支撐的建議）

## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/pipeline_doc.md", "w") as f:
    f.write(report)
print("✅ 報告已存到 output/pipeline_doc.md")


---
## Section 7：Next Step 規劃

> 填寫 `docs/upgrade_plan.md`，想一下後續課程學的工具怎麼套回這條 pipeline。
> 這份加上 pipeline_doc.md 就是你課後 push 到 GitHub 的基礎。


In [ ]:
# （不需要改）
checks = [("pipeline.db", "SQLite 資料庫"), ("processed", "統計結果"), ("output/pipeline_doc.md", "顧問報告")]
print("📋 產出確認：")
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {desc}: {path}")
    if not exists: all_ok = False
if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ {t}: {n} 筆")
        except: print(f"  ❌ {t} 不存在"); all_ok = False
    c.close()
print("\n🎉 全部完成！" if all_ok else "\n⚠️ 有缺漏。")
print("\n📋 接下來：README + upgrade_plan + 2 分鐘 Demo + Section 9-10（回家作業）")


---
## Section 8：FastAPI — 把分析結果變成 API

> 別人不能打開 .db 檔。API 把結果包裝成網址。
> 🅰️ 在下面寫 / 🅱️ 開 `api.py`（solution）


### 🎯 完成這段後你應該有：
- `api.py` 能跑起來
- 瀏覽器打開 `http://localhost:8000/health` 回 200

> ⏱ 卡住了？`api.py` 已經是 solution，改好路徑直接跑即可。


### Step 8-1 🔴 較難：定義 endpoint

> 💡 **範例**：假設要把學生成績做成 API：
> ```python
> from fastapi import FastAPI
> api = FastAPI(title="成績查詢 API")
>
> @api.get("/top_students")
> def top_students():
>     c = sqlite3.connect("school.db")
>     df = pd.read_sql("SELECT name, score FROM students ORDER BY score DESC LIMIT 10", c)
>     c.close()
>     return df.to_dict(orient="records")
> ```
> 🎯 定義 endpoint：`/health`、`/stats/products`（商品排行）、`/stats/countries`（各國統計）、`/analyzed`（LLM 結果）


In [ ]:
# TODO 🔴: FastAPI
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

api = FastAPI(title="零售銷售分析 API")

# /health
@api.get("/health")
def health():
    return {"status": "ok"}

# TODO: /stats/products — 從 cleaned_orders 查商品排行


# TODO: /analyzed — 從 analyzed_orders 查 LLM 結果


print("✅ API 定義完成")


### Step 8-2 🟡 中等：啟動 + 測試

> 💡 **範例**：
> ```python
> resp = requests.get("http://localhost:8000/top_students")
> print(resp.json())  # [{name: "Alice", score: 95}, ...]
> ```
> 🎯 測試你定義的 3 個 endpoint


In [ ]:
# 啟動（直接跑）
import threading, uvicorn, time
thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host":"0.0.0.0","port":8000,"log_level":"warning"})
thread.daemon = True
thread.start()
time.sleep(2)
print("✅ API 已啟動")

# TODO 🟡: 用 requests 測試
import requests


> 🅱️ `api.py` 是 solution（6 endpoint）。本地跑：`uvicorn api:app --reload --port 8000`


---
## Section 9（回家作業）：Dashboard

> 🅰️ 用 ipywidgets（Colab 可跑）/ 🅱️ 開 `app.py`（Streamlit，solution）


### Step 9-1 🔴 較難：互動 Dashboard

> 💡 **範例**：做一個「選班級 → 看成績分佈」的互動：
> ```python
> import ipywidgets as widgets
> from IPython.display import display, clear_output
> import matplotlib.pyplot as plt
>
> class_dropdown = widgets.Dropdown(options=["全部", "A班", "B班"], description="選班級：")
>
> def update(class_name):
>     clear_output(wait=True)
>     display(class_dropdown)
>     data = students if class_name == "全部" else students[students["class"] == class_name]
>     print(f"{class_name}: {len(data)} 人, 平均 {data['score'].mean():.1f}")
>     data["score"].hist(bins=20)
>     plt.show()
>
> widgets.interact(update, class_name=class_dropdown)
> ```
> 🎯 做一個「選國家 → 看商品銷售排行」的互動


In [ ]:
# TODO 🔴: 互動 Dashboard
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# 讀資料
df_dash = pd.read_sql("SELECT * FROM cleaned_orders", conn)

# TODO: 建立下拉選單


# TODO: 定義更新函式（篩選 → 統計 → 畫圖）


# TODO: 綁定互動


> 🅱️ `app.py` 是 solution。本地跑：`streamlit run app.py`


---
## Section 10（回家作業）：本地部署指引

```bash
cd data/raw/topic_1
uvicorn api:app --reload --port 8000    # Terminal 1
streamlit run app.py                     # Terminal 2
```

| 現在 | 升級後 | 對應課程 |
|------|--------|---------|
| SQLite | MySQL / BigQuery | 資料庫模組 |
| 手動跑 | Airflow DAG | Airflow 模組 |
| 本地 Streamlit | Docker 容器化 | Docker 模組 |
| 本地開發 | GCP 雲端部署 | GCP 模組 |
